In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import os, gc
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')
os.environ['XGBOOST_WARN_PREDICT_FALLBACK'] = '0'

%matplotlib inline

In [2]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e10/train.csv
/kaggle/input/competitions/playground-series-s6e10/test.csv


In [3]:
path = '/kaggle/input/competitions/playground-series-s6e10'
df_train = pd.read_csv(path+'/train.csv')
df_test = pd.read_csv(path+'/test.csv')
df_submission = pd.read_csv(path+'/sample_submission.csv')

TARGET = 'satisfaction'
ID = 'id'
# EXPERIMENT_NAME = "XGBoost"
df_train[TARGET] = df_train[TARGET].astype(int)
NJ_SEED = 42

for df in [df_train, df_test]:
    df['Arrival Delay in Minutes'] = df['Arrival Delay in Minutes'].fillna(df['Departure Delay in Minutes'])

CAT_FEATURES = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
RATING_COLS = [
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking', 
    'Food and drink', 'Online boarding', 'Seat comfort', 'Inflight entertainment', 
    'On-board service', 'Leg room service', 'Baggage handling', 'Checkin service', 'Cleanliness'
]

# 2.  FEATURE ENGINEERING

In [4]:
# 🧮 Generating Domain Features & Pairs
for df in [df_train, df_test]:
    df['biz_elite_nj'] = ((df['Type of Travel'] == 'Business travel') & (df['Class'] == 'Business')).astype(np.int8)
    df['Wifi_Booking_pair'] = df['Inflight wifi service'].astype(str) + "_" + df['Ease of Online booking'].astype(str)
    df['Wifi_Time_pair']    = df['Inflight wifi service'].astype(str) + "_" + df['Departure/Arrival time convenient'].astype(str)
    df['Boarding_Wifi_pair']= df['Online boarding'].astype(str) + "_" + df['Inflight wifi service'].astype(str)
    df['Class_Travel_pair'] = df['Class'].astype(str) + "_" + df['Type of Travel'].astype(str)
    df['Cust_Travel_pair']  = df['Customer Type'].astype(str) + "_" + df['Type of Travel'].astype(str)
    df['Class_Cust_pair']   = df['Class'].astype(str) + "_" + df['Customer Type'].astype(str)
    df['Class_Wifi_pair']     = df['Class'].astype(str) + "_" + df['Inflight wifi service'].astype(str)
    df['Class_Boarding_pair'] = df['Class'].astype(str) + "_" + df['Online boarding'].astype(str)
    df['Travel_Wifi_pair']    = df['Type of Travel'].astype(str) + "_" + df['Inflight wifi service'].astype(str)
    df['Travel_Boarding_pair']= df['Type of Travel'].astype(str) + "_" + df['Online boarding'].astype(str)
    df['Seat_Ent_pair']       = df['Seat comfort'].astype(str) + "_" + df['Inflight entertainment'].astype(str)
    df['Route_Class_pair'] = df['Flight Distance'].astype(str) + "_" + df['Class'].astype(str)
    df['Route_Travel_pair'] = df['Flight Distance'].astype(str) + "_" + df['Type of Travel'].astype(str)

INTERACTION_COLS = [
    'Wifi_Booking_pair', 'Wifi_Time_pair', 'Boarding_Wifi_pair', 'Class_Travel_pair', 'Cust_Travel_pair', 
    'Class_Cust_pair', 'Class_Wifi_pair', 'Class_Boarding_pair', 'Travel_Wifi_pair', 'Travel_Boarding_pair', 
    'Seat_Ent_pair', 'Route_Class_pair', 'Route_Travel_pair'
]

# 🔢 Extracting  Digits
DIGIT_MAP = {'Age': [0, 1], 'Flight Distance': [0, 1, 2, 3], 'Departure Delay in Minutes': [1], 'Arrival Delay in Minutes': [1]}
digit_feats_nj = [] 
for c, powers in DIGIT_MAP.items():
    for k in powers:
        col_name = f"{c}_digit{k}"
        df_train[col_name] = (df_train[c].fillna(0) // (10**k) % 10).astype(np.int8)
        df_test[col_name] = (df_test[c].fillna(0) // (10**k) % 10).astype(np.int8)
        digit_feats_nj.append(col_name)

# ✈️ Extracting Route Profiles based on Flight Distance
for df in [df_train, df_test]:
    df['Route_ID'] = df['Flight Distance'].astype(int)

route_counts = pd.concat([df_train['Route_ID'], df_test['Route_ID']], ignore_index=True).value_counts()
for df in [df_train, df_test]:
    df['Route_Freq'] = df['Route_ID'].map(route_counts).astype(np.int32)

route_profile_cols = ['Age', 'Departure Delay in Minutes'] + RATING_COLS
route_profile = df_train.groupby('Route_ID')[route_profile_cols].mean()
route_profile.columns = [f"Route_Mean_{c.replace(' ', '_').replace('/', '_')}" for c in route_profile.columns]
route_profile = route_profile.reset_index()

df_train = df_train.merge(route_profile, on="Route_ID", how="left")
df_test  = df_test.merge(route_profile,  on="Route_ID", how="left")

for c in route_profile.columns:
    if c == "Route_ID": continue
    df_train[c] = df_train[c].fillna(df_train[c].mean())
    df_test[c]  = df_test[c].fillna(df_train[c].mean())

df_train.drop(columns=['Route_ID'], inplace=True)
df_test.drop(columns=['Route_ID'], inplace=True)

# Setup Encoding
TARGET_ENCODE_COLS = CAT_FEATURES + RATING_COLS + digit_feats_nj + INTERACTION_COLS

num_to_cat_cols = []
for c in ['Age', 'Flight Distance']: 
    cat_name = f"{c}_cat"
    df_train[cat_name] = df_train[c].fillna(-1).astype(str)
    df_test[cat_name] = df_test[c].fillna(-1).astype(str)
    num_to_cat_cols.append(cat_name)

TARGET_ENCODE_COLS.extend(num_to_cat_cols)

for df in [df_train, df_test]:
    for col in CAT_FEATURES + digit_feats_nj:
        df[col] = df[col].astype(str)
    for col in RATING_COLS:
        df[col] = "Rating_" + df[col].astype(str)

# 📊 Applying Frequency Encoding
combined_nj = pd.concat([df_train, df_test], ignore_index=True)
for col in TARGET_ENCODE_COLS:
    freq_mapping = combined_nj[col].value_counts(normalize=True).to_dict()
    df_train[f"{col}_FE"] = df_train[col].map(freq_mapping).astype(np.float32)
    df_test[f"{col}_FE"]  = df_test[col].map(freq_mapping).astype(np.float32)
del combined_nj; gc.collect()

# Hardcoded Pruning List
FEATURES_TO_DROP = [
    'Departure Delay in Minutes_digit1_FE', 'Departure Delay in Minutes_digit1_TE_auto',
    'Arrival Delay in Minutes_digit1_FE', 'Arrival Delay in Minutes_digit1_TE_auto',
    'Departure Delay in Minutes', 'Flight Distance_digit3_FE', 'Flight Distance_digit1_FE',
    'Age_digit0_FE', 'Flight Distance_digit0_FE', 'Flight Distance_digit2_FE'
]

In [5]:
df_train

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Class_Cust_pair_FE,Class_Wifi_pair_FE,Class_Boarding_pair_FE,Travel_Wifi_pair_FE,Travel_Boarding_pair_FE,Seat_Ent_pair_FE,Route_Class_pair_FE,Route_Travel_pair_FE,Age_cat_FE,Flight Distance_cat_FE
0,0,36,694,Rating_4,Rating_4,Rating_4,3,Rating_1,Rating_4,Rating_1,...,0.359224,0.076735,0.094671,0.038736,0.056338,0.071492,0.000527,0.000345,0.022100,0.000674
1,1,56,651,Rating_5,Rating_5,Rating_5,5,Rating_5,Rating_4,Rating_4,...,0.426533,0.079811,0.197423,0.105471,0.245176,0.069587,0.001349,0.002527,0.018407,0.004160
2,2,31,1371,Rating_3,Rating_4,Rating_2,5,Rating_3,Rating_3,Rating_3,...,0.359224,0.143177,0.136935,0.096820,0.086142,0.117335,0.000315,0.000222,0.016230,0.000539
3,3,10,1616,Rating_3,Rating_5,Rating_3,3,Rating_3,Rating_3,Rating_3,...,0.359224,0.143177,0.136935,0.096820,0.086142,0.117335,0.000283,0.000259,0.004326,0.000953
4,4,23,481,Rating_3,Rating_4,Rating_3,4,Rating_5,Rating_3,Rating_5,...,0.108644,0.143177,0.136935,0.166667,0.127787,0.174994,0.000480,0.000468,0.023795,0.000738
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
699630,699630,57,612,Rating_2,Rating_1,Rating_1,1,Rating_1,Rating_3,Rating_3,...,0.359224,0.144391,0.136935,0.171270,0.127787,0.021392,0.003005,0.002806,0.018999,0.004509
699631,699631,30,862,Rating_2,Rating_2,Rating_2,3,Rating_4,Rating_2,Rating_4,...,0.108644,0.144391,0.123992,0.171270,0.093184,0.195023,0.006208,0.005242,0.021711,0.009577
699632,699632,49,3148,Rating_5,Rating_5,Rating_5,5,Rating_4,Rating_4,Rating_4,...,0.426533,0.079811,0.197423,0.105471,0.245176,0.069587,0.000069,0.000069,0.023008,0.000069
699633,699633,61,679,Rating_5,Rating_2,Rating_2,2,Rating_5,Rating_4,Rating_4,...,0.359224,0.029046,0.094671,0.105471,0.245176,0.069587,0.004528,0.003600,0.005906,0.006286


In [6]:
df_train.duplicated().sum()

np.int64(0)

In [7]:
df_train.isna().sum()

id                                   0
Age                                  0
Flight Distance                      0
Inflight wifi service                0
Departure/Arrival time convenient    0
                                    ..
Seat_Ent_pair_FE                     0
Route_Class_pair_FE                  0
Route_Travel_pair_FE                 0
Age_cat_FE                           0
Flight Distance_cat_FE               0
Length: 101, dtype: int64

In [8]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699635 entries, 0 to 699634
Columns: 101 entries, id to Flight Distance_cat_FE
dtypes: float32(39), float64(15), int32(1), int64(6), int8(1), object(39)
memory usage: 427.7+ MB


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelBinarizer, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.preprocessing import RobustScaler
import time
from sklearn.model_selection import StratifiedKFold

from xgboost import XGBClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
import optuna

In [10]:
FEATURES = [c for c in df_train.columns if c not in [ID, TARGET, 'Arrival_Delay_missing'] + TARGET_ENCODE_COLS]
X = df_train[FEATURES + TARGET_ENCODE_COLS]
y = df_train[TARGET]

X_test = df_test[FEATURES + TARGET_ENCODE_COLS]

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=NJ_SEED)
oof_preds = np.zeros(len(df_train))
test_preds = np.zeros(len(df_test))
fold_scores = []
best_iters = []

In [11]:
def prepare_fold_data(X_train, y_train, X_valid, X_test_fold=None):
        
    te_auto = TargetEncoder(shuffle=True, cv=5, smooth='auto', random_state=NJ_SEED)
    X_train_enc = te_auto.fit_transform(X_train[TARGET_ENCODE_COLS], y_train)
    X_valid_enc = te_auto.transform(X_valid[TARGET_ENCODE_COLS])
    # X_test_enc  = te_auto.transform(X_test_fold[TARGET_ENCODE_COLS])
    
    for i, col in enumerate(TARGET_ENCODE_COLS):
        X_train[f"{col}_TE_auto"] = X_train_enc[:, i].astype(np.float32)
        X_valid[f"{col}_TE_auto"] = X_valid_enc[:, i].astype(np.float32)
        # X_test_fold[f"{col}_TE_auto"] = X_test_enc[:, i].astype(np.float32)
        X_train.drop(columns=[col], inplace=True)
        X_valid.drop(columns=[col], inplace=True)
        # X_test_fold.drop(columns=[col], inplace=True)
    # X_test_enc
    del X_train_enc, X_valid_enc ; gc.collect()
    
    X_train.drop(columns=FEATURES_TO_DROP, inplace=True, errors='ignore')
    X_valid.drop(columns=FEATURES_TO_DROP, inplace=True, errors='ignore')
    # X_test_fold.drop(columns=FEATURES_TO_DROP, inplace=True, errors='ignore')

    return X_train, X_valid #, X_test_fold

In [12]:
import cupy as cp
X_train,X_valid,y_train, y_valid = train_test_split(X, y, test_size=0.2, shuffle=True, stratify=y)
X_train, X_valid = prepare_fold_data(X_train, y_train, X_valid)
X_train_gpu = cp.asarray(X_train.values)
X_valid_gpu = cp.asarray(X_valid.values)
y_train_gpu = cp.asarray(y_train.values)

In [ ]:
import gc
import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import TargetEncoder
from sklearn.metrics import roc_auc_score


# ============================================================
# 1. CONFIGURATION
# ============================================================

NJ_SEED = 42
N_SPLITS = 5
N_TRIALS = 50

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=NJ_SEED
)


# ============================================================
# 2. TARGET ENCODING ET PRÉPARATION DES FOLDS
# ============================================================

def prepare_fold_data(X_train, y_train, X_valid):

    X_train = X_train.copy()
    X_valid = X_valid.copy()

    # Encodage des variables catégorielles ciblées
    te = TargetEncoder(
        shuffle=True,
        cv=5,
        smooth="auto",
        random_state=NJ_SEED
    )

    X_train_enc = te.fit_transform(
        X_train[TARGET_ENCODE_COLS],
        y_train
    )

    X_valid_enc = te.transform(
        X_valid[TARGET_ENCODE_COLS]
    )

    # Remplacer les variables originales par leurs versions encodées
    for i, col in enumerate(TARGET_ENCODE_COLS):

        encoded_col = f"{col}_TE_auto"

        X_train[encoded_col] = X_train_enc[:, i].astype(np.float32)
        X_valid[encoded_col] = X_valid_enc[:, i].astype(np.float32)

        X_train.drop(columns=[col], inplace=True)
        X_valid.drop(columns=[col], inplace=True)

    # Suppression des variables inutiles
    X_train.drop(
        columns=FEATURES_TO_DROP,
        inplace=True,
        errors="ignore"
    )

    X_valid.drop(
        columns=FEATURES_TO_DROP,
        inplace=True,
        errors="ignore"
    )

    # Garantir le même ordre et les mêmes colonnes
    X_valid = X_valid.reindex(
        columns=X_train.columns,
        fill_value=0
    )

    del X_train_enc, X_valid_enc
    gc.collect()

    return X_train, X_valid


# ============================================================
# 3. FONCTION OBJECTIVE D'OPTUNA
# ============================================================

def objective(trial):

    

    fold_scores = []

    # Validation croisée externe
    for fold, (train_idx, valid_idx) in enumerate(
        skf.split(X, y), start=1
    ):

        # Séparation du fold
        X_tr = X.iloc[train_idx].copy()
        y_tr = y.iloc[train_idx]

        X_val = X.iloc[valid_idx].copy()
        y_val = y.iloc[valid_idx]

        # Encodage effectué uniquement à partir du fold d'entraînement
        X_tr, X_val = prepare_fold_data(
            X_tr, y_tr, X_val
        )

        # Entraînement LightGBM
        model = lgb.LGBMClassifier(**params)

        model.fit(X_tr, y_tr)

        # Probabilités de la classe positive
        y_pred = model.predict_proba(X_val)[:, 1]

        # Calcul du ROC-AUC
        score = roc_auc_score(y_val, y_pred)
        fold_scores.append(score)

        print(
            f"Trial {trial.number} | "
            f"Fold {fold}/{N_SPLITS} | "
            f"ROC-AUC: {score:.6f}"
        )

        del X_tr, X_val, y_pred, model
        gc.collect()

    mean_auc = float(np.mean(fold_scores))

    # Enregistrer les résultats détaillés du trial
    trial.set_user_attr("fold_scores", fold_scores)
    trial.set_user_attr("std_auc", float(np.std(fold_scores)))

    return mean_auc


# ============================================================
# 4. LANCEMENT DE L'OPTIMISATION
# ============================================================

study = optuna.create_study(
    direction="maximize",
    study_name="LightGBM_Airline_Satisfaction",
    sampler=optuna.samplers.TPESampler(seed=NJ_SEED)
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    gc_after_trial=True
)


# ============================================================
# 5. AFFICHAGE DES MEILLEURS RÉSULTATS
# ============================================================

print("\n" + "=" * 60)
print("MEILLEURS RÉSULTATS OPTUNA - LIGHTGBM")
print("=" * 60)

print(f"Meilleur ROC-AUC moyen : {study.best_value:.6f}")

print("\nMeilleurs hyperparamètres :")
for param, value in study.best_params.items():
    print(f"{param}: {value}")

print("\nScores des folds du meilleur trial :")
print(study.best_trial.user_attrs["fold_scores"])

print(
    "Écart-type des scores : "
    f"{study.best_trial.user_attrs['std_auc']:.6f}"
)


# Paramètres à réutiliser pour l'entraînement final
best_lgbm_params = {
    **study.best_params,
    "objective": "binary",
    "metric": "auc",
    "random_state": NJ_SEED,
    "n_jobs": -1,
    "verbosity": -1,
    "force_col_wise": True,
    "subsample_freq": 1,
}

print("\nDictionnaire best_lgbm_params prêt à être réutilisé.")

In [13]:
# ============================================================
# 3. OBJECTIVE OPTUNA
# ============================================================

def objective(trial):

    fold_scores = []

    # Hyperparamètres à optimiser
    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "random_state": NJ_SEED,
        "n_jobs": -1,
        "verbosity": 0,

        "n_estimators": trial.suggest_int(
            "n_estimators", 500, 2500, step=250
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 3, 10
        ),

        "min_child_weight": trial.suggest_float(
            "min_child_weight", 1, 20, log=True
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),

        "gamma": trial.suggest_float(
            "gamma", 0.0, 5.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-4, 10.0, log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 20.0, log=True
        ),
    }
    
        # Entraînement XGBoost
    model = xgb.XGBClassifier(**params)

    model.fit(
        X_train_gpu,
        y_train_gpu,
        verbose=True
    )

    # Prédictions de probabilités
    valid_probs = model.predict_proba(
        X_valid_gpu
    )[:, 1]

    # ROC-AUC du pli
    fold_auc = roc_auc_score(
        y_valid, valid_probs
    ) 

    return fold_auc


# ============================================================
# 4. LANCEMENT DE L'OPTIMISATION
# ============================================================

study = optuna.create_study(
    direction="maximize",
    study_name="XGBoost_Airline_Satisfaction",
    sampler=optuna.samplers.TPESampler(
        seed=NJ_SEED
    )
)

study.optimize(
    objective,
    n_trials=150,
    gc_after_trial=True,
    show_progress_bar=True, n_jobs=-1
)


# ============================================================
# 5. RESULTATS
# ============================================================

print("\nMeilleur ROC-AUC moyen :", study.best_value)
best_trial = study.best_trial
print("\nMeilleurs hyperparamètres :")
for param, value in best_trial.items():
    print(f"{param}: {value}")

[I 2026-10-27 09:14:18,049] A new study created in memory with name: XGBoost_Airline_Satisfaction


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2026-10-27 09:19:29,528] Trial 3 finished with value: 0.9581172803222149 and parameters: {'n_estimators': 1250, 'learning_rate': 0.07987441266065891, 'max_depth': 10, 'min_child_weight': 2.1013134128287754, 'subsample': 0.9208012919271529, 'colsample_bytree': 0.8278216673712259, 'gamma': 0.6820766102019454, 'reg_alpha': 0.00013474284375921333, 'reg_lambda': 0.12805523170129807}. Best is trial 3 with value: 0.9581172803222149.
[I 2026-10-27 09:19:32,307] Trial 0 finished with value: 0.9601853294912595 and parameters: {'n_estimators': 1750, 'learning_rate': 0.058200092784191935, 'max_depth': 6, 'min_child_weight': 16.86455872947408, 'subsample': 0.9142720998593701, 'colsample_bytree': 0.9787670987104586, 'gamma': 1.1762637350936793, 'reg_alpha': 1.3238582832509103, 'reg_lambda': 0.017001926322993726}. Best is trial 0 with value: 0.9601853294912595.
[I 2026-10-27 09:19:42,453] Trial 1 finished with value: 0.9603170178715901 and parameters: {'n_estimators': 2250, 'learning_rate': 0.0571

AttributeError: 'FrozenTrial' object has no attribute 'items'

[I 2026-10-26 23:19:20,858] Trial 99 finished with value: 0.9611511137398503 and parameters: {'n_estimators': 1750, 'learning_rate': 0.017683520882721834, 'max_depth': 9, 'min_child_weight': 1.9341362690580444, 'subsample': 0.9190370723234387, 'colsample_bytree': 0.6094183952589787, 'gamma': 2.4152960082346953, 'reg_alpha': 0.003631396524511289, 'reg_lambda': 0.5876633279512555}. Best is trial 99 with value: 0.9611511137398503.

In [14]:
import optuna.visualization as vis

display(vis.plot_param_importances(study))
display(vis.plot_optimization_history(study))

In [ ]:
# Dictionnaire réutilisable pour le modèle final
xgb_params = {
    # 'n_estimators': 1750, 'learning_rate': 0.017683520882721834
    # , 'max_depth': 9, 'min_child_weight': 1.9341362690580444
    # , 'subsample': 0.9190370723234387, 'colsample_bytree': 0.6094183952589787
    # , 'gamma': 2.4152960082346953, 'reg_alpha': 0.003631396524511289
    # , 'reg_lambda': 0.5876633279512555, "objective": "binary:logistic"
    **study.best_params,"objective": "binary:logistic"
    , "eval_metric": "auc", "tree_method": "hist"
    , "random_state": NJ_SEED, "n_jobs": -1
    , "verbosity": 0
}

xgb_params

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=NJ_SEED)
for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, y_train = X.iloc[train_idx].copy(), y.iloc[train_idx]
    X_valid, y_valid = X.iloc[valid_idx].copy(), y.iloc[valid_idx]
    X_test_fold = X_test.copy()

    te_auto = TargetEncoder(shuffle=True, cv=5, smooth='auto', random_state=NJ_SEED)
    X_train_enc = te_auto.fit_transform(X_train[TARGET_ENCODE_COLS], y_train)
    X_valid_enc = te_auto.transform(X_valid[TARGET_ENCODE_COLS])
    X_test_enc  = te_auto.transform(X_test_fold[TARGET_ENCODE_COLS])

    for i, col in enumerate(TARGET_ENCODE_COLS):
        X_train[f"{col}_TE_auto"] = X_train_enc[:, i].astype(np.float32)
        X_valid[f"{col}_TE_auto"] = X_valid_enc[:, i].astype(np.float32)
        X_test_fold[f"{col}_TE_auto"] = X_test_enc[:, i].astype(np.float32)
        X_train.drop(columns=[col], inplace=True)
        X_valid.drop(columns=[col], inplace=True)
        X_test_fold.drop(columns=[col], inplace=True)

    del X_train_enc, X_valid_enc, X_test_enc; gc.collect()
    
    X_train.drop(columns=FEATURES_TO_DROP, inplace=True, errors='ignore')
    X_valid.drop(columns=FEATURES_TO_DROP, inplace=True, errors='ignore')
    X_test_fold.drop(columns=FEATURES_TO_DROP, inplace=True, errors='ignore')

    model = xgb.XGBClassifier(**xgb_params)
    
    model.fit(
        X_train, y_train, 
        eval_set=[(X_valid, y_valid)], 
        # cp.asarray(X_train), cp.asarray(y_train), 
        # eval_set=[(cp.asarray(X_valid), cp.asarray(y_valid))], 
        verbose=1000,
    )

    
    valid_probs = model.predict_proba(X_valid)[:, 1]
    oof_preds[valid_idx] = valid_probs
    test_preds += model.predict_proba(X_test_fold)[:, 1] / skf.n_splits
    
    fold_auc = roc_auc_score(y_valid, valid_probs)
    fold_scores.append(fold_auc)
    best_iter = model.best_iteration if hasattr(model, 'best_iteration') else 0
    best_iters.append(best_iter)
    print(f"   --> Fold {fold} | Best Iter: {best_iter} | ROC-AUC: {fold_auc:.5f}")

In [ ]:
mean_auc = np.mean(fold_scores)
print("\n" + "-"*40)
print(f"🏆 FINAL CV: {mean_auc:.5f} ± {np.std(fold_scores):.5f}")
print("-" * 40)

In [ ]:
readable_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
print(readable_time)
df_submission[TARGET] = test_preds
df_submission.to_csv(f"submission-{readable_time}.csv", index=False)
print("💾 CV Results and Artifacts saved.")

In [ ]:
br

In [ ]:
estimators = [
    ('XGBoost Classifier', XGBClassifier(**xgb_params, random_state=42))
    ,('LGBM Classifier', LGBMClassifier(**lgbm_params, random_state=42, force_row_wise=True,verbosity=-1))
]


model = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(random_state=42)
)

In [ ]:
scores = []
models = {}
for fold, (train_idx, test_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train.iloc[train_idx], y_train[train_idx]
    X_ts, y_ts = X_train.iloc[test_idx], y_train[test_idx]


    start_time = time.perf_counter()
    model.fit(X_tr, y_tr)
    # models[f'model-{fold}']=model
    
    end_time = time.perf_counter()

    # Durée totale
    training_time = end_time - start_time

    # predicting validation score
    y_pred = model.predict_proba(X_ts)[:, 1]
    score = roc_auc_score(y_ts, y_pred)
    scores.append(score)
    
    # df_submission[f'satisfaction_{fold}']= model.predict_proba(_df_test)[:, 1]
    
    print(f'============================ fold: {fold+1}, score: {score}, time: {training_time:.2f} secondes ==================')
    

In [ ]:
np.mean(scores)